# 01 — Stagnone di Marsala: Domain & Mesh Evaluation

This notebook evaluates the Model B mesh (`Stagnone_dxy01_15m_net.nc`), assesses boundary adequacy,
and determines if domain extension is needed for the turbidity modeling component.

**Key questions:**
1. Is the mesh resolution adequate at the lagoon inlets?
2. Are the offshore boundaries far enough for CMEMS forcing?
3. Does the domain need northward extension to capture the Trapani canal turbidity source (37.996°N)?
4. What is the bathymetry quality inside the lagoon?

## 1. Imports

In [ ]:
%matplotlib inline
import os
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
import xarray as xr
import xugrid as xu
import dfm_tools as dfmt

project_root = Path(r'F:\StagnoneDT')
model_b_dir = project_root / 'oldModel' / 'Stagnone_py_lagoon3D.dsproj_data' / 'Stagnone_dxy01_15m' / 'input'
bathy_file = project_root / 'oldModel' / 'bat_stagnone_20m_2006.xyz'

## 2. Load and inspect the mesh

In [ ]:
# Load Model B mesh
netfile = model_b_dir / 'Stagnone_dxy01_15m_net.nc'
uds = xu.open_dataset(str(netfile))

# Extract node coordinates
node_x = uds.grid.node_x
node_y = uds.grid.node_y

print('=== MESH STATISTICS ===')
print(f'Nodes: {node_x.size}')
print(f'Longitude range: {float(node_x.min()):.4f} to {float(node_x.max()):.4f}')
print(f'Latitude range:  {float(node_y.min()):.4f} to {float(node_y.max()):.4f}')

# Approximate domain size
lon_range = float(node_x.max()) - float(node_x.min())
lat_range = float(node_y.max()) - float(node_y.min())
print(f'\nDomain extent:')
print(f'  Longitude: {lon_range:.3f}° ≈ {lon_range * 111 * np.cos(np.radians(37.86)):.1f} km')
print(f'  Latitude:  {lat_range:.3f}° ≈ {lat_range * 111:.1f} km')

# Check bathymetry
if 'mesh2d_node_z' in uds:
    z = uds['mesh2d_node_z']
    print(f'\nBathymetry (mesh2d_node_z):')
    print(f'  Range: {float(z.min()):.2f} to {float(z.max()):.2f} m')
    print(f'  Mean:  {float(z.mean()):.2f} m')
    print(f'  NaN count: {int(np.isnan(z.values).sum())}')

In [ ]:
# Compute cell sizes (edge lengths)
edge_nodes = uds.grid.edge_node_connectivity
node_x_arr = np.asarray(node_x)
node_y_arr = np.asarray(node_y)

x1 = node_x_arr[edge_nodes[:, 0]]
y1 = node_y_arr[edge_nodes[:, 0]]
x2 = node_x_arr[edge_nodes[:, 1]]
y2 = node_y_arr[edge_nodes[:, 1]]

# Approximate edge lengths in meters (WGS84)
dx_deg = x2 - x1
dy_deg = y2 - y1
dx_m = dx_deg * 111000 * np.cos(np.radians(37.86))
dy_m = dy_deg * 111000
edge_lengths_m = np.sqrt(dx_m**2 + dy_m**2)

print('=== EDGE LENGTH STATISTICS ===')
print(f'  Min:    {edge_lengths_m.min():.1f} m')
print(f'  Max:    {edge_lengths_m.max():.1f} m')
print(f'  Mean:   {edge_lengths_m.mean():.1f} m')
print(f'  Median: {np.median(edge_lengths_m):.1f} m')
print(f'  P10:    {np.percentile(edge_lengths_m, 10):.1f} m')
print(f'  P90:    {np.percentile(edge_lengths_m, 90):.1f} m')

# Histogram of edge lengths
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(edge_lengths_m, bins=100, edgecolor='black', linewidth=0.5)
ax.set_xlabel('Edge length (m)')
ax.set_ylabel('Count')
ax.set_title('Distribution of mesh edge lengths')
ax.axvline(50, color='r', ls='--', label='50m (target lagoon resolution)')
ax.axvline(200, color='orange', ls='--', label='200m (target offshore resolution)')
ax.legend()
plt.tight_layout()
plt.savefig(str(project_root / 'figures' / 'edge_length_histogram.png'), dpi=150)
plt.show()

## 3. Evaluate inlet resolution

The northern inlet is very narrow (~0.3m depth, critical for lagoon exchange).
We need at minimum 5-10 cells across the inlet width.

In [ ]:
# Plot zoom on northern inlet (Boca Nord ~37.905°N, 12.457°E)
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Northern inlet
ax = axes[0]
ax.set_title('Northern Inlet (Boca Nord)')
if 'mesh2d_node_z' in uds:
    uds.mesh2d_node_z.ugrid.plot(ax=ax, center=False, cmap='terrain', vmin=-3, vmax=1)
uds.grid.plot(ax=ax, linewidth=0.5, color='black', alpha=0.4)
dfmt.plot_coastlines(ax=ax, crs='EPSG:4326')
ax.set_xlim(12.44, 12.47)
ax.set_ylim(37.895, 37.915)
ax.plot(12.457, 37.905, 'r*', markersize=15, label='BocaNord station')
ax.legend()

# Southern inlet
ax = axes[1]
ax.set_title('Southern Inlet (Boca Sud)')
if 'mesh2d_node_z' in uds:
    uds.mesh2d_node_z.ugrid.plot(ax=ax, center=False, cmap='terrain', vmin=-3, vmax=1)
uds.grid.plot(ax=ax, linewidth=0.5, color='black', alpha=0.4)
dfmt.plot_coastlines(ax=ax, crs='EPSG:4326')
ax.set_xlim(12.43, 12.47)
ax.set_ylim(37.84, 37.86)
ax.plot(12.449, 37.847, 'r*', markersize=15, label='BocaSud station')
ax.legend()

fig.tight_layout()
plt.savefig(str(project_root / 'figures' / 'inlet_resolution.png'), dpi=150)
plt.show()

## 4. Evaluate boundary extent

Check how far offshore the boundaries are and whether they capture the turbidity source areas.

In [ ]:
# Load boundary polyline
pli_file = model_b_dir / 'Stagnone_dxy01_15m.pli'
with open(pli_file, 'r') as f:
    lines = f.readlines()

# Parse PLI file (skip header lines)
bnd_points = []
for line in lines:
    parts = line.strip().split()
    if len(parts) >= 2:
        try:
            x, y = float(parts[0]), float(parts[1])
            if 10 < x < 15 and 35 < y < 40:  # reasonable lon/lat range
                bnd_points.append((x, y))
        except ValueError:
            pass

bnd_x = [p[0] for p in bnd_points]
bnd_y = [p[1] for p in bnd_points]

print(f'Boundary polyline: {len(bnd_points)} points')
print(f'  Lon range: {min(bnd_x):.4f} to {max(bnd_x):.4f}')
print(f'  Lat range: {min(bnd_y):.4f} to {max(bnd_y):.4f}')

# Lagoon center for distance reference
lagoon_lon, lagoon_lat = 12.46, 37.87
offshore_dist_km = (lagoon_lon - min(bnd_x)) * 111 * np.cos(np.radians(37.86))
print(f'\nApprox. offshore distance (west): {offshore_dist_km:.1f} km')

# Turbidity source locations
src1 = (12.468, 37.917)  # Airport canal
src2 = (12.508, 37.996)  # Trapani canal
print(f'\nTurbidity sources:')
print(f'  Source 1 (Airport canal): {src1} — in domain: {min(bnd_y) < src1[1] < max(bnd_y)}')
print(f'  Source 2 (Trapani canal): {src2} — in domain: {min(bnd_y) < src2[1] < max(bnd_y)}')

In [ ]:
# Full domain overview with boundary, turbidity sources, and observation stations
fig, ax = plt.subplots(figsize=(12, 10))

# Plot mesh
if 'mesh2d_node_z' in uds:
    uds.mesh2d_node_z.ugrid.plot(ax=ax, center=False, cmap='terrain', vmin=-100, vmax=5, alpha=0.7)
uds.grid.plot(ax=ax, linewidth=0.2, color='gray', alpha=0.3)

# Plot boundary
ax.plot(bnd_x, bnd_y, 'r-', linewidth=2, label='Open boundary')

# Plot turbidity sources
ax.plot(*src1, 'v', color='brown', markersize=12, label='Source 1: Airport canal')
ax.plot(*src2, 'v', color='darkred', markersize=12, label='Source 2: Trapani canal (OUTSIDE domain)')

# Load and plot observation stations
obs_file = model_b_dir / 'Stagnone_dxy01_15m_obs.xyn'
obs_df = pd.read_csv(obs_file, sep=r'\s+', header=None, names=['x', 'y', 'name'])
ax.scatter(obs_df['x'], obs_df['y'], c='blue', s=40, zorder=5, label=f'Observation stations ({len(obs_df)})')

# Annotate key stations
for _, row in obs_df.iterrows():
    if row['name'] in ['BocaNord', 'BocaSud', 'AltaVilaEst']:
        ax.annotate(row['name'], (row['x'], row['y']), fontsize=8, fontweight='bold',
                    xytext=(5, 5), textcoords='offset points')

dfmt.plot_coastlines(ax=ax, crs='EPSG:4326')
ax.set_title('Stagnone Model B — Domain Overview')
ax.legend(loc='lower left')

fig.tight_layout()
plt.savefig(str(project_root / 'figures' / 'domain_overview.png'), dpi=150, bbox_inches='tight')
plt.show()

## 5. Load and evaluate 2006 bathymetry

In [ ]:
# Load the 2006 bathymetry XYZ
bathy_2006 = pd.read_csv(str(bathy_file), sep=r'\s+', header=None, names=['x', 'y', 'z'])
print(f'2006 Bathymetry: {len(bathy_2006)} points')
print(f'  X range: {bathy_2006.x.min():.1f} to {bathy_2006.x.max():.1f}')
print(f'  Y range: {bathy_2006.y.min():.1f} to {bathy_2006.y.max():.1f}')
print(f'  Z range: {bathy_2006.z.min():.2f} to {bathy_2006.z.max():.2f}')
print(f'  Z mean:  {bathy_2006.z.mean():.2f}')

# Convention check
print(f'\n=== CONVENTION ANALYSIS ===')
print(f'  All z >= 0: {(bathy_2006.z >= 0).all()}')
print(f'  → 2006 data uses POSITIVE-DOWN convention (depth below surface)')
print(f'  → Model mesh uses POSITIVE-UP convention (elevation, negative = below sea level)')
print(f'  → To use in model: z_model = -z_2006')

# Determine CRS from coordinate ranges
if bathy_2006.x.max() > 1000:
    print(f'\n  Coordinate system: UTM (values > 1000)')
    print(f'  Likely EPSG:32632 (UTM Zone 32N)')
else:
    print(f'\n  Coordinate system: Geographic (WGS84)')

In [ ]:
# Plot 2006 bathymetry with CORRECTED sign (negate to match model convention)
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Original (positive-down)
ax = axes[0]
sc = ax.scatter(bathy_2006.x, bathy_2006.y, c=bathy_2006.z, cmap='viridis_r',
                s=0.5, vmin=0, vmax=5)
plt.colorbar(sc, ax=ax, label='Depth below surface (m)')
ax.set_title('2006 Survey — original (positive-down)')
ax.set_aspect('equal')

# Corrected (positive-up, model convention)
ax = axes[1]
sc = ax.scatter(bathy_2006.x, bathy_2006.y, c=-bathy_2006.z, cmap='terrain',
                s=0.5, vmin=-5, vmax=15)
plt.colorbar(sc, ax=ax, label='Elevation (m, model convention)')
ax.set_title('2006 Survey — corrected (z_model = -z_2006)')
ax.set_aspect('equal')

fig.suptitle('bat_stagnone_20m_2006.xyz — Convention Comparison', fontsize=14)
fig.tight_layout()
plt.savefig(str(project_root / 'figures' / 'bathymetry_2006_convention.png'), dpi=150)
plt.show()

print('NOTE: When integrating 2006 data into the model, negate z values.')
print('  Lagoon areas: z_2006 ~ 0.5-2.0 → z_model ~ -0.5 to -2.0')
print('  Land areas:   z_2006 ~ 0 or negative (unlikely) → z_model ~ 0 or positive')

## 6. Assessment and recommendations

### Boundary adequacy
- Offshore boundary ~10-15 km west of lagoon — **adequate for hydrodynamics**
- Source 1 (Airport canal, 37.917°N) — check if inside domain
- Source 2 (Trapani canal, 37.996°N) — **likely OUTSIDE domain**, needs extension

### Mesh resolution
- Check the edge length statistics and inlet zoom plots above
- Target: minimum 5-10 cells across inlet width at both Boca Nord and Boca Sud

### Recommended actions
1. **Phase 1 (now):** Use current mesh as-is to get the model running
2. **Phase 2 (later):** Extend domain northward to ~38.01°N for turbidity modeling
3. **Phase 3 (later):** Refine inlet resolution if needed based on calibration results